# ИУ5-64Б Марянян Артур Вариант 14

### Задача №2.
Для заданного набора данных проведите обработку пропусков в данных для одного категориального и одного количественного признака. Какие способы обработки пропусков в данных для категориальных и количественных признаков Вы использовали? Какие признаки Вы будете использовать для дальнейшего построения моделей машинного обучения и почему?
Для студентов группы ИУ5-64Б - для произвольной колонки данных построить график "Скрипичная диаграмма (violin plot)".

## 1. Подключение библиотек

Загружаем NuGet-пакеты, совместимые с текущей средой .NET 10.

**Используемые пакеты:**
- `Microsoft.Data.Analysis` (0.22.1) — работа с табличными данными
- `Microsoft.ML` (3.0.1) — инструменты ML.NET
- `Plotly.NET.Interactive` (5.0.0) и `Plotly.NET.CSharp` (0.12.0) — построение графиков
- `Microsoft.DotNet.Interactive.Formatting` (1.0.0-beta.25070.1) — улучшенное отображение в ноутбуке

In [2]:
#r "nuget: Microsoft.Data.Analysis, 0.22.1"
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.12.0"
#r "nuget: Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1"

Installed Packages Microsoft.Data.Analysis, 0.22.1 Microsoft.DotNet.Interactive.Formatting, 1.0.0-beta.25070.1 Microsoft.ML, 3.0.1 Plotly.NET.CSharp, 0.12.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\artur\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\artur\.nuget\packages\microsoft.data.analysis\0.22.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

## 2. Импорт пространств имён

Подключаем необходимые пространства имён для удобной работы с данными и визуализацией.

In [3]:
using Microsoft.Data.Analysis;
using Microsoft.ML;
using Plotly.NET;
using PlotlyNET = Plotly.NET.CSharp.Chart;  // алиас для краткости
using System;
using System.Linq;
using System.Globalization;
using System.IO;

## 3. Загрузка набора данных

Файл `Admission_Predict.csv` должен находиться в одной папке с ноутбуком. Загружаем его в DataFrame, явно указывая разделитель и культуру (точка как десятичный разделитель).

In [10]:
var dataPath = Path.Combine(Directory.GetCurrentDirectory(), "Data/Admission_Predict_Ver1.1.csv");

string[] columnNames = new[] { "Serial No.", "GRE Score", "TOEFL Score", "University Rating", "SOP", "LOR ", "CGPA", "Research", "Chance of Admit " };
Type[] dataTypes = new[] { typeof(int), typeof(float), typeof(float), typeof(float), typeof(float), typeof(float), typeof(double), typeof(int), typeof(double) };

var rawData = DataFrame.LoadCsv(dataPath,
    separator: ',',
    header: true,
    columnNames: columnNames,
    dataTypes: dataTypes,
    cultureInfo: CultureInfo.InvariantCulture);

rawData.Head(5).Display();

index,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
0,1,337,118,4,4.5,4.5,9.65,1,0.92
1,2,324,107,4,4,4.5,8.87,1,0.76
2,3,316,104,3,3,3.5,8,1,0.72
3,4,322,110,3,3.5,2.5,8.67,1,0.8
4,5,314,103,2,2,3,8.21,0,0.65


## 4. Искусственное внесение пропусков

В исходном датасете пропусков нет, поэтому для демонстрации методов обработки мы внесём ~10% пропусков в количественный признак `CGPA` и ~5% — в категориальный `University Rating`.

In [11]:
var rnd = new Random(42);
for (int i = 0; i < rawData.Rows.Count; i++)
{
    // CGPA - количественный
    if (rnd.NextDouble() < 0.1)
        rawData["CGPA"][i] = null;
    // University Rating - категориальный (числовой код)
    if (rnd.NextDouble() < 0.05)
        rawData["University Rating"][i] = null;
}

Console.WriteLine("Количество пропусков после внесения:");
foreach (var col in rawData.Columns)
{
    // Используем Cast<object?>(), так как object? допускает null
    int nullCount = col.Cast<object?>().Count(v => v == null);
    Console.WriteLine($"{col.Name}: {nullCount}");
}

Количество пропусков после внесения:
Serial No.: 0
GRE Score: 0
TOEFL Score: 0
University Rating: 37
SOP: 0
LOR : 0
CGPA: 50
Research: 0
Chance of Admit : 0


## 5. Обработка пропусков

- **Количественный признак** — `CGPA`: замена средним арифметическим (Mean Imputation).
- **Категориальный признак** — `University Rating`: замена наиболее частым значением – модой (Most Frequent Value).

Вычислим моду и среднее по доступным значениям, затем заполним все `null` в DataFrame.

In [12]:
// Мода для University Rating (тип float?)
var ratingValues = rawData["University Rating"].Cast<float?>().Where(v => v.HasValue).Select(v => v.Value);
var ratingMode = ratingValues.GroupBy(x => x)
                              .OrderByDescending(g => g.Count())
                              .First().Key;
Console.WriteLine($"Мода University Rating: {ratingMode}");

// Среднее для CGPA
var cgpaValues = rawData["CGPA"].Cast<double?>().Where(v => v.HasValue).Select(v => v.Value);
var cgpaMean = cgpaValues.Average();
Console.WriteLine($"Среднее CGPA: {cgpaMean:F3}");

// Заполнение пропусков
for (int i = 0; i < rawData.Rows.Count; i++)
{
    if (rawData["University Rating"][i] == null)
        rawData["University Rating"][i] = ratingMode;
    if (rawData["CGPA"][i] == null)
        rawData["CGPA"][i] = cgpaMean;
}

// Проверка
Console.WriteLine("\nПосле обработки:");
foreach (var col in rawData.Columns)
{
    int nullCount = col.Cast<object?>().Count(v => v == null);
    Console.WriteLine($"{col.Name}: {nullCount}");
}

Мода University Rating: 3
Среднее CGPA: 8,586

После обработки:
Serial No.: 0
GRE Score: 0
TOEFL Score: 0
University Rating: 0
SOP: 0
LOR : 0
CGPA: 0
Research: 0
Chance of Admit : 0


## 6. Ответы на вопросы

**Какие способы обработки пропусков в данных для категориальных и количественных признаков Вы использовали?**
- Для количественного признака `CGPA` применена **замена средним арифметическим** (Mean Imputation).
- Для категориального признака `University Rating` применена **замена модой** (Most Frequent Value), то есть подстановка наиболее часто встречающегося значения.

**Какие признаки Вы будете использовать для дальнейшего построения моделей машинного обучения и почему?**
- Целевая переменная: `Chance of Admit`.
- Признаки, которые войдут в модель: `GRE Score`, `TOEFL Score`, `University Rating`, `SOP`, `LOR`, `CGPA`, `Research`.
- `Serial No.` исключён, так как это просто идентификатор строки, не несущий информации.
- Все оставленные признаки имеют прямую связь с вероятностью поступления: они отражают академическую успеваемость (CGPA), результаты стандартизированных тестов (GRE, TOEFL), престиж текущего университета (University Rating), качество мотивационного письма (SOP), рекомендаций (LOR) и наличие исследовательского опыта (Research). Это ключевые факторы, влияющие на решение приёмной комиссии.

## 7. Скрипичная диаграмма (violin plot) — задание для ИУ5-64Б

Построим скрипичную диаграмму для произвольного столбца. В качестве примера возьмём `CGPA` (средний балл после заполнения пропусков). На диаграмме отображаются плотность распределения, медиана, межквартильный размах, а также линия среднего.

In [ ]:
// Извлечение значений CGPA (после заполнения пропусков)
double[] cgpaArray = rawData["CGPA"]
    .Cast<double?>()
    .Where(d => d.HasValue)
    .Select(d => d.Value)
    .ToArray();

// Построение скрипичной диаграммы
var violinChart = PlotlyNET.Violin<string, double, string>(
    X: Enumerable.Repeat("CGPA", cgpaArray.Length).ToArray(),
    Y: cgpaArray
)
.WithTitle("Скрипичная диаграмма среднего балла (CGPA)")
.WithSize(800, 500);

display(violinChart);

<!-- Plotly chart will be drawn inside this DIV -->